In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(r"C:\Users\patil\OneDrive\Documents\ai-hiring-bias-xai\data\raw/AI_Resume_Screening.csv")
print(df.shape)
df.head()

(1000, 11)


,Resume_ID,Name,Skills,Experience (Years),Education,Certifications,Job Role,Recruiter Decision,Salary Expectation ($),Projects Count,AI Score (0-100)
0,1,Ashley Ali,"TensorFlow, NLP, Pytorch",10,B.Sc,NaN,AI Researcher,Hire,104895,8,100
1,2,Wesley Roman,"Deep Learning, Machine Learning, Python, SQL",10,MBA,Google ML,Data Scientist,Hire,113002,1,100
2,3,Corey Sanchez,"Ethical Hacking, Cybersecurity, Linux",1,MBA,Deep Learning Specialization,Cybersecurity Analyst,Hire,71766,7,70
3,4,Elizabeth Carney,"Python, Pytorch, TensorFlow",7,B.Tech,AWS Certified,AI Researcher,Hire,46848,0,95
4,5,Julie Hill,"SQL, React, Java",4,PhD,NaN,Software Engineer,Hire,87441,9,100


In [3]:
df["Recruiter Decision"].unique()

array(['Hire', 'Reject'], dtype=object)

In [4]:
df = df.rename(columns={
    "Experience (Years)": "experience_years",
    "AI Score (0-100)": "ai_score",
    "Recruiter Decision": "hired",
    "Projects Count": "projects_count",
    "Salary Expectation ($)": "salary_expectation"
})

In [5]:
df["hired"] = (
    df["hired"]
    .astype(str)
    .str.strip()
    .str.lower()
)

In [6]:
df["hired"] = df["hired"].map({
    "yes": 1,
    "no": 0,
    "selected": 1,
    "rejected": 0,
    "hire": 1,
    "not hire": 0
})

In [7]:
df["hired"].value_counts(dropna=False)

hired
1.0    812
NaN    188
Name: count, dtype: int64

In [8]:
np.random.seed(42)
df["gender"] = np.random.choice(
    ["Male", "Female"],
    size=len(df)
)

In [9]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

cat_cols = df.select_dtypes(include=["object"]).columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [10]:
# Number of samples
n = len(df)

# Create synthetic negative samples
df_negative = df.sample(n // 2, random_state=42).copy()
df_negative["hired"] = 0

# Combine original + negative samples
df = pd.concat([df, df_negative], ignore_index=True)

# Verify
print("\nAfter fix:")
print(df["hired"].value_counts())


After fix:
hired
1.0    1000
0.0     500
Name: count, dtype: int64


In [11]:
df.to_csv(r"C:\Users\patil\OneDrive\Documents\ai-hiring-bias-xai/data/processed/resumes_cleaned.csv", index=False)
print(df.shape)
print("DATASET SAVED SUCCESSFULLY")

(1500, 12)
DATASET SAVED SUCCESSFULLY
